In [2]:
import numpy as np
import json
import pickle
from pathlib import Path

import nltk
from nltk.corpus import gutenberg

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense, Input
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [3]:
SEED = 42
VOCAB_SIZE = 12000
MAX_CONTEXT = 12         
EMBEDDING_DIM = 64
LSTM_UNITS = 96
BATCH_SIZE = 128
EPOCHS = 30

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [5]:
# 1. Download and load the dataset

nltk.download("gutenberg", quiet=True)
raw_text = gutenberg.raw("shakespeare-hamlet.txt").lower()
lines = [line.strip() for line in raw_text.splitlines() if line.strip()]

split_at = int(len(lines) * 0.90)
train_lines = lines[:split_at]
val_lines = lines[split_at:]

In [6]:
# 2. Fit tokenizer only on training text. OOV handles unseen validation words.

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>", filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(train_lines)

def make_examples(text_lines):
    X, y = [], []
    for line in text_lines:
        tokens = tokenizer.texts_to_sequences([line])[0]
        # Each prefix predicts the next word. Use at most MAX_CONTEXT prior tokens.
        for i in range(1, len(tokens)):
            context = tokens[max(0, i - MAX_CONTEXT):i]
            target = tokens[i]
            if 0 < target < VOCAB_SIZE:
                X.append(context)
                y.append(target)
    X = pad_sequences(X, maxlen=MAX_CONTEXT, padding="pre", truncating="pre")
    return np.asarray(X, dtype=np.int32), np.asarray(y, dtype=np.int32)

X_train, y_train = make_examples(train_lines)
X_val, y_val = make_examples(val_lines)

if len(X_train) == 0 or len(X_val) == 0:
    raise ValueError("Not enough training/validation examples. Check the dataset and preprocessing.")

In [9]:
# 3. Build the model. 

model = Sequential([
    Input(shape=(MAX_CONTEXT,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, mask_zero=True),
    LSTM(LSTM_UNITS),
    Dropout(0.25),
    Dense(VOCAB_SIZE, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
]


In [10]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 12, 64)              │         768,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 96)                  │          61,824 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 96)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 12000)               │       1,164,000 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,993,824 (7.61 MB)

 Trainable params: 1,993,824 (7.61 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# 4. Train and validate

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/30
183/183 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 7.7640 - sparse_categorical_accuracy: 0.0309 - val_loss: 7.1091 - val_sparse_categorical_accuracy: 0.0432 - learning_rate: 0.0010
Epoch 2/30
183/183 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 6.7094 - sparse_categorical_accuracy: 0.0334 - val_loss: 7.0829 - val_sparse_categorical_accuracy: 0.0428 - learning_rate: 0.0010
Epoch 3/30
183/183 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 6.5535 - sparse_categorical_accuracy: 0.0335 - val_loss: 7.0886 - val_sparse_categorical_accuracy: 0.0428 - learning_rate: 0.0010
Epoch 4/30
182/183 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 6.4841 - sparse_categorical_accuracy: 0.0351
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
183/183 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 6.4595 - sparse_categorical_accuracy: 0.0350 - val_loss: 7.1392 - val_sparse_categorical_accuracy: 0.0469 - learning_rate: 0.0010
Epoch 5/30
183/183 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 6.38

In [12]:
# 5. Save all inference artifacts

model.save(ARTIFACTS / "next_word_lstm.keras")
with open(ARTIFACTS / "tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open(ARTIFACTS / "config.json", "w", encoding="utf-8") as f:
    json.dump({"max_context": MAX_CONTEXT, "vocab_size": VOCAB_SIZE}, f, indent=2)

print(f"\nSaved model and tokenizer to: {ARTIFACTS.resolve()}")
print(f"Training examples: {len(X_train):,}; validation examples: {len(X_val):,}")
print("Note: next-word prediction is inherently difficult; top-k predictions can be more useful than one word.")


Saved model and tokenizer to: C:\Users\bhupi\OneDrive\Desktop\Next-Word Prediction using LSTM\artifacts
Training examples: 23,303; validation examples: 2,429
Note: next-word prediction is inherently difficult; top-k predictions can be more useful than one word.
